# 03_model_baseline.ipynb
## Baseline Modified BKT Training (with Forgetting)

*Model*: Standard pyBKT + forgets=True (your first extra parameter).  
*Purpose*: Establish a strong baseline before adding hints & behavior tuning (next notebook).


In [7]:
# ================================================
# Imports & Load Data
# ================================================
import pandas as pd
from pyBKT.models import Model
from pathlib import Path

DATA_PROCESSED = Path("../data/processed")
DATA_PROCESSED.mkdir(parents=True, exist_ok=True)

df = pd.read_csv("../data/raw/synthetic_autism_data.csv")

# Keep pyBKT-required column names for evaluation compatibility
df_bkt = df.rename(columns={
    "anon_student_id": "user_id",
    "correct": "correct"
})[["user_id", "skill_name", "correct"]]

print("Baseline data ready")
print("Columns:", df_bkt.columns.tolist())

Baseline data ready
Columns: ['user_id', 'skill_name', 'correct']


In [8]:
# ================================================
# Train Baseline Model with Forgetting
# ================================================
model = Model(seed=42, num_fits=5)

model.fit(
    data=df_bkt,
    skills=list(df["skill_name"].unique()),
    forgets=True           # baseline extension: allow forgetting
)

print("✅ Baseline model fitted with forgetting")
print("\nLearned Parameters (first few):")
print(model.params().head())

c:\Users\HP\.pyenv\pyenv-win\versions\3.10.11\lib\site-packages\pyBKT\fit\M_step.py:61: RuntimeWarning: invalid value encountered in divide
  model['pi_0'] = init_softcounts[:] / np.sum(init_softcounts[:])


✅ Baseline model fitted with forgetting

Learned Parameters (first few):
                           value
skill    param   class          
counting prior   default     NaN
         learns  default 1.00000
         guesses default 0.50000
         slips   default 0.50000
         forgets default 0.00000


In [10]:
# ================================================
# Evaluation
# ================================================
import numpy as np
from sklearn.metrics import accuracy_score, mean_squared_error, roc_auc_score

pred_df = model.predict(data=df_bkt)

# Find a prediction-probability column exposed by pyBKT
candidate_pred_cols = [
    "correct_predictions", "prediction", "predictions", "y_hat",
    "correct_prediction", "correct_pred", "pred"
]
pred_col = next((c for c in candidate_pred_cols if c in pred_df.columns), None)
if pred_col is None:
    numeric_cols = [
        c for c in pred_df.columns
        if c != "correct" and np.issubdtype(pred_df[c].dtype, np.number)
    ]
    if not numeric_cols:
        raise ValueError("No numeric prediction column found in model.predict output")
    pred_col = numeric_cols[0]

eval_df = pred_df.copy()
if "correct" not in eval_df.columns:
    eval_df["correct"] = df_bkt["correct"].values

eval_df = eval_df[["correct", pred_col]].dropna()
if eval_df.empty:
    raise ValueError("No valid rows available for evaluation after dropping NaNs")

y_true = eval_df["correct"].astype(int)
y_prob = eval_df[pred_col].astype(float)
y_pred = (y_prob >= 0.5).astype(int)

metrics = {
    "accuracy": float(accuracy_score(y_true, y_pred)),
    "rmse": float(np.sqrt(mean_squared_error(y_true, y_prob)))
}
if y_true.nunique() > 1:
    metrics["auc"] = float(roc_auc_score(y_true, y_prob))
else:
    metrics["auc"] = float("nan")

print("Baseline Performance:")
print(metrics)

# Save model
MODELS_DIR = Path("../models")
MODELS_DIR.mkdir(parents=True, exist_ok=True)
import pickle
with open(MODELS_DIR / "baseline_model.pkl", "wb") as f:
    pickle.dump(model, f)

print("✅ Baseline saved to ../models/baseline_model.pkl")

Baseline Performance:
{'accuracy': 0.5104493464052288, 'rmse': 0.6996789646650606, 'auc': 0.5}
✅ Baseline saved to ../models/baseline_model.pkl


### Next Steps
- Notebook 04 will tune the full extended model (α_h for hints + β_b for behavior).
- This baseline already shows the benefit of your forgetting parameter.